In [11]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import amfinder_load as AmfLoad
import amfinder_config as AmfConfig
import amfinder_model as AmfModel
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np

# Auxillary Functions

In [12]:

def get_predictions(x, model, device="cpu", num_classes=6, batch_size=32):
        model.eval()
        y = len(x)*[np.zeros(num_classes)]
        inference_dataset = AmfLoad.CustomNormalisedDataset(x, y)
        test_loader = DataLoader(inference_dataset, batch_size=batch_size, shuffle=False)

        all_y_probs = []

        with torch.no_grad():  
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(device)
                outputs = model(x_batch)
                probabilities = torch.softmax(outputs, dim=1)
                all_y_probs.append(probabilities.cpu().numpy())

        all_y_probs = np.vstack(all_y_probs)

        return all_y_probs

def validate_transfer(keras_model, pytorch_model):
    x = get_random_image()
    import random
    x = random.sample(x,1)
    test_input = [i.transpose(2, 1, 0) for i in x]
    test_input = np.array(test_input)

    print("\nValidating weight transfer...")
    
    print(f"Input shape for Keras (NHWC format): {test_input.shape}")
    
    # Feed the NHWC format input to Keras model
    # Note: keras.predict() automatically disables dropout
    keras_output = keras_model.predict(test_input/255)
    
    # Set PyTorch model to evaluation mode (disables dropout) and get prediction
    pytorch_model.eval()
    torch_output = get_predictions(x, pytorch_model)
    print(f"PyTorch output shape: {torch_output.shape}")
    
    print("keras output", keras_output)
    print("pytorch output", torch_output)
    
    return 

def get_random_image():
    input_files = AmfConfig.find_files_in_directory("../../Training Datasets/126_Training_csv")
    parts = AmfLoad.categorise_path(input_files)
    train_images = parts["train"]
    dataset_loader_test = AmfLoad.TileFilesandData(train_images)
    x, y, fn, rows, cols = dataset_loader_test.get_all_data()
    return x


# AmfModel create_cnn1() function

In [13]:
import numpy as np
import torch
from tensorflow import keras
import tensorflow as tf

def transfer_weights_keras_to_modular_pytorch(keras_model_path, pytorch_model):
    """
    Transfer weights from a Keras .h5 model to the modular PyTorch model.
    
    Args:
        keras_model_path (str): Path to the Keras .h5 model file
        pytorch_model (torch.nn.Module): PyTorch model to load weights into
        
    Returns:
        The PyTorch model with loaded weights
    """
    # Load Keras model
    keras_model = keras.models.load_model(keras_model_path)
    
    # Dictionary of Keras layers by name
    keras_layers = {layer.name: layer for layer in keras_model.layers}
    
    # Transfer convolutional layers
    for keras_name, attr_name in [
        ('C11', 'conv11'), ('C12', 'conv12'), ('C13', 'conv13'),
        ('C21', 'conv21'), ('C22', 'conv22'),
        ('C31', 'conv31'), ('C32', 'conv32'), ('C4', 'conv4')
    ]:
        if keras_name in keras_layers:
            # Get weights from Keras
            w, b = keras_layers[keras_name].get_weights()
            
            # Convert from Keras format (height, width, in_channels, out_channels)
            # to PyTorch format (out_channels, in_channels, height, width)
            w = np.transpose(w, (3, 2, 0, 1))
            
            # Set to PyTorch layer (now in the conv submodule)
            layer = getattr(pytorch_model.conv, attr_name)
            layer.weight.data = torch.FloatTensor(w)
            layer.bias.data = torch.FloatTensor(b)
    
    # Get dimensions from the model
    # Run a forward pass with a dummy input to get the flattened size
    with torch.no_grad():
        dummy_input = torch.zeros(1, 3, 126, 126)
        _, features = pytorch_model.conv(dummy_input)
        feature_size = features.size(1)
    
    # Handle the first FC layer which comes after flattening
    if 'FCRS1' in keras_layers:
        w, b = keras_layers['FCRS1'].get_weights()
        
        # For the modular architecture, we need to be especially careful with reshaping
        # We need to match the exact flattening order used in the ConvolutionalBlocks class
        
        # First, we need to determine the feature map dimensions after the last maxpool
        # This should be 256 channels with 5x5 spatial dimensions as in the original model
        
        # Reshape to match original feature map structure
        c, h, w_dim = 256, 5, 5  # Expected dimensions for 126x126 input
        
        # Verify dimensions match
        assert feature_size == c * h * w_dim, f"Feature size mismatch. Got {feature_size}, expected {c * h * w_dim}"
        
        # Reshape the weights to match the tensor that would be flattened
        # Keras uses channel-last format (H, W, C)
        w_reshaped = w.reshape(h, w_dim, c, w.shape[1])
        
        # Transpose to PyTorch's channel-first format (C, H, W)
        w_transposed = np.transpose(w_reshaped, (2, 0, 1, 3))
        
        # Flatten spatial dimensions to match PyTorch's nn.Flatten() behavior
        w_pytorch = w_transposed.reshape(c * h * w_dim, w.shape[1]).T
        
        # Set to PyTorch layer (now in the fc submodule)
        pytorch_model.fc.fc1.weight.data = torch.FloatTensor(w_pytorch)
        pytorch_model.fc.fc1.bias.data = torch.FloatTensor(b)
    
    # Handle other fully connected layers
    for keras_name, attr_name in [('FCRS2', 'fc2'), ('RS', 'output')]:
        if keras_name in keras_layers:
            w, b = keras_layers[keras_name].get_weights()
            
            # Transpose for PyTorch (output_features, input_features)
            w = w.T
            
            # Set to PyTorch layer (now in the fc submodule)
            layer = getattr(pytorch_model.fc, attr_name)
            layer.weight.data = torch.FloatTensor(w)
            layer.bias.data = torch.FloatTensor(b)
    
    return pytorch_model

def verify_model_equivalence(keras_model_path, pytorch_model, test_input=None, rtol=1e-5, atol=1e-5):
    """
    Verify that the Keras and PyTorch models produce equivalent outputs.
    
    Args:
        keras_model_path: Path to the Keras model
        pytorch_model: The PyTorch model
        test_input: Optional test input, default is random
        rtol: Relative tolerance for comparison
        atol: Absolute tolerance for comparison
        
    Returns:
        bool: True if outputs are equivalent, False otherwise
    """
    # Load Keras model
    keras_model = keras.models.load_model(keras_model_path)
    
    # Create random test input if not provided
    if test_input is None:
        # Create random input of the right shape
        test_input = np.random.randint(0, 256, (1, 126, 126, 3)).astype(np.float32) / 255.0
    
    # Get Keras prediction
    keras_output = keras_model.predict(test_input)
    
    # Convert test input for PyTorch (NHWC to NCHW)
    pytorch_input = np.transpose(test_input, (0, 3, 1, 2))
    pytorch_input_tensor = torch.FloatTensor(pytorch_input)
    
    # Get PyTorch prediction
    pytorch_model.eval()
    with torch.no_grad():
        pytorch_output = pytorch_model(pytorch_input_tensor)
        
        # If the output isn't already a softmax, apply softmax
        # if hasattr(pytorch_model.fc, 'activation') and pytorch_model.fc.activation is None:
        pytorch_output = torch.nn.functional.softmax(pytorch_output, dim=1)
        
        pytorch_output = pytorch_output.numpy()
    
    # Compare outputs
    is_close = np.allclose(keras_output, pytorch_output, rtol=rtol, atol=atol)
    
    if is_close:
        print("✓ Models produce equivalent outputs!")
    else:
        print("✗ Models produce different outputs.")
        print(f"  Maximum absolute difference: {np.max(np.abs(keras_output - pytorch_output))}")
        print(f"  Keras output shape: {keras_output.shape}, PyTorch output shape: {pytorch_output.shape}")
        print(f"  Keras output: {keras_output}")
        print(f"  PyTorch output: {pytorch_output}")
    
    return is_close




# Path to your Keras model
keras_model_path = '../../CNN1v2.h5'

# Create PyTorch model and transfer weights
pytorch_model = AmfModel.create_cnn1()
pytorch_model = transfer_weights_keras_to_modular_pytorch(keras_model_path, pytorch_model)

# Verify models produce the same output
keras_model = tf.keras.models.load_model(keras_model_path)
validate_transfer(keras_model, pytorch_model)

verify_model_equivalence(keras_model_path, pytorch_model)

c:\Users\d80120\AppData\Local\anaconda3\envs\root_fungal_colonisation_env\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


[14:01:23] Tile extraction.
Dropping questions from annotations for ABE784_C3_h_Default_Extended_1
[14:01:23] 1 images in filtered dataset.

Validating weight transfer...
Input shape for Keras (NHWC format): (1, 126, 126, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step


c:\Users\d80120\AppData\Local\anaconda3\envs\root_fungal_colonisation_env\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


PyTorch output shape: (1, 3)
keras output [[0.00243429 0.02351314 0.97405255]]
pytorch output [[0.00223424 0.02198328 0.9757825 ]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
✓ Models produce equivalent outputs!


True

# Validation

In [14]:
import random
x = get_random_image()
x = random.sample(x,1)
x = np.array(x)
x = x/255
x_torch = torch.tensor(x, dtype = torch.float32)
x_keras = np.transpose(x, (0,2,3,1))
print(x_torch.shape)
torch_output = torch.softmax(pytorch_model(x_torch), dim=1)
keras_output = keras_model.predict(x_keras)

print("torch output: ", torch_output)
print("keras output: ", keras_output)


[14:01:24] Tile extraction.
Dropping questions from annotations for ABE784_C3_h_Default_Extended_1
[14:01:24] 1 images in filtered dataset.
torch.Size([1, 3, 126, 126])
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
torch output:  tensor([[0.0268, 0.8834, 0.0898]], grad_fn=<SoftmaxBackward0>)
keras output:  [[0.02680713 0.8834341  0.08975879]]


In [ ]:
# torch.save(pytorch_model,"../../CNN1v2_evangelisti_weights_pytorch_3_classes.pth")

# 3 to 6 output classes

In [6]:
def modify_output_size(model, new_output_size=6):
    """
    Modify the output size of an existing CNN1 model.
    
    Args:
        model: Existing PyTorch CNN1 model
        new_output_size: Number of output classes (default: 6)
    
    Returns:
        Modified PyTorch model with new output size
    """
    # Create a new FCLayers module with the desired output size
    new_fc = AmfModel.FCLayers(
        fc_in_size=model.fc.fc1.in_features,
        label='RS',
        output_size=new_output_size
    )
    
    # Copy weights for shared layers (fc1 and fc2)
    new_fc.fc1.weight.data = model.fc.fc1.weight.data.clone()
    new_fc.fc1.bias.data = model.fc.fc1.bias.data.clone()
    new_fc.fc2.weight.data = model.fc.fc2.weight.data.clone()
    new_fc.fc2.bias.data = model.fc.fc2.bias.data.clone()
    
    # The output layer is initialized fresh with kaiming_uniform_
    # The initialization is already handled in the FCLayers._initialise_weights method
    
    # Replace the fc module in the model
    model.fc = new_fc
    
    return model

new_model = modify_output_size(pytorch_model)

In [ ]:
# torch.save(new_model,"../../CNN1v2_evangelisti_weights_pytorch_6_classes.pth")

In [8]:
import torch

def verify_weights_preserved(original_model, modified_model):
    """
    Verify that weights are identical before the output layer.
    
    Args:
        original_model: The original model before modification
        modified_model: The model after modifying output size
        
    Returns:
        bool: True if all weights (except output) are identical
    """
    # Extract state dicts
    original_state = original_model.state_dict()
    modified_state = modified_model.state_dict()
    
    # Names of parameters to check
    # We exclude the output layer parameters
    output_layer_keys = ['fc.output.weight', 'fc.output.bias']
    
    # Check each parameter
    all_preserved = True
    mismatches = []
    
    for key in original_state:
        # Skip output layer
        if key in output_layer_keys:
            continue
            
        # If parameter exists in both models
        if key in modified_state:
            # Check if weights are identical
            is_equal = torch.allclose(original_state[key], modified_state[key])
            
            if not is_equal:
                all_preserved = False
                mismatches.append(key)
    
    # Print results
    if all_preserved:
        print("✓ All weights before output layer are preserved!")
    else:
        print("✗ Some weights are different:")
        for key in mismatches:
            print(f"  - {key}")
    
    return all_preserved

# Example usage

# Create original model
original_model = pytorch_model

# Save a copy of the original model
original_state = original_model.state_dict()

# Modify output size
modified_model = new_model

# Verify weights are preserved
verify_weights_preserved(original_model, modified_model)

✓ All weights before output layer are preserved!


True